# 01: Parameterization Workflow

Learn UV coordinates for a 3D surface and project the intensity volume onto that UV map — using the high-level `Parameterizer` API.

This mirrors the per-frame logic of the legacy `nuvo_projection_timeseries.py`, but in a handful of lines.

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from pathlib import Path

from seamless import Parameterizer

## Load a synthetic volume

The synthetic file stores a 4D microscopy mock-up `(T, Z, Y, X)`. The `topology` attribute names the geometry; ellipsoids are closed surfaces, so we map them to the `'cylinder'` Nuvo topology.

In [ ]:
seg_file = Path('data/synthetic/ellipsoid.h5')

with h5py.File(seg_file, 'r') as f:
    volumes = f['microscopy_mockup'][:]            # (T, Z, Y, X)
    file_topology = f.attrs.get('topology', 'ellipsoid')

# Map the geometry name to a Nuvo topology string.
NUVO_TOPOLOGY = {'ellipsoid': 'cylinder', 'cylinder': 'cylinder', 'bent_sheet': 'bent_sheet'}
topology = NUVO_TOPOLOGY.get(file_topology, 'cylinder')

print(f'Volumes: {volumes.shape}  (T, Z, Y, X)')
print(f'File topology: {file_topology}  ->  Nuvo topology: {topology}')

## Build a `Parameterizer` from the first frame

`Parameterizer.from_volume` runs the surface-extraction stage for you: binarize at `threshold` → keep the largest connected component → marching cubes → random subsample. The point cloud is normalized internally and surface normals are estimated automatically.

In [ ]:
t0 = 0
param = Parameterizer.from_volume(
    volumes[t0],
    topology=topology,
    threshold=43,             # synthetic mock-up: background < 43 < interior
    num_target_points=20_000,
)

print(f'Surface points: {param.xyz.shape[0]}')
print(f'Device: {param.device}')

## Train the NuvoMLP (t = 0)

Full training on the first frame. Iteration counts come from `ParameterizationConfig` (default 300 at t=0); pass `iterations=` to override. This takes a couple of minutes on CPU/MPS.

In [ ]:
model = param.train(verbose=True)

uv_map = param.get_uv_map()              # (N, 2) UV coordinates of the surface points
print(f'UV map: {uv_map.shape}')
print(f'u in [{uv_map[:, 0].min():.2f}, {uv_map[:, 0].max():.2f}]',
      f'v in [{uv_map[:, 1].min():.2f}, {uv_map[:, 1].max():.2f}]')

## Visualize the learned UV map

Each 3D surface point now has a 2D `(u, v)` coordinate. Coloring by depth (Z) shows how the surface is unrolled onto the unit square.

In [ ]:
pts = param.points_voxel.numpy()         # surface points in voxel space

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(uv_map[:, 0], uv_map[:, 1], c=pts[:, 0], s=3, cmap='viridis')
ax.set(xlabel='u', ylabel='v', title='Learned UV map (colored by Z)')
fig.colorbar(sc, ax=ax, label='Z (voxel)')
plt.show()

## Project the volume onto UV space

`project_to_uv_along_normals` samples the volume along the surface normal at several offsets, producing a multi-layer image stack. The max-projection across layers is a clean 2D cartographic view of the surface.

In [ ]:
multilayer = param.project_to_uv_along_normals(volumes[t0], uv_res=512)
max_projection = multilayer.max(axis=0)

print(f'Multilayer stack: {multilayer.shape}  (layers, H, W)')
print(f'Max projection:   {max_projection.shape}')

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(multilayer[multilayer.shape[0] // 2], cmap='gray')
axes[0].set_title('Mid normal-offset layer')
axes[1].imshow(max_projection, cmap='gray')
axes[1].set_title('Max projection')
for a in axes:
    a.axis('off')
plt.show()

## Time-series warm-start

For a sequence, reuse the trained network as the starting point for the next frame — full training at `t = 0`, then a short fine-tune for each subsequent frame via `base_model=`. This is the core trick of the legacy timeseries script and is far faster than training every frame from scratch.

In [ ]:
n_frames = min(12, volumes.shape[0])
projections = [max_projection]
base = model

for t in range(1, n_frames):
    p_t = Parameterizer.from_volume(volumes[t], topology=topology,
                                    threshold=43, num_target_points=20_000)
    p_t.train(base_model=base)           # fine-tune from the previous frame (50 iters)
    base = p_t.get_model()
    proj = p_t.project_to_uv_along_normals(volumes[t], uv_res=512).max(axis=0)
    projections.append(proj)
    print(f't={t}: projection {proj.shape}')

fig, axes = plt.subplots(1, len(projections), figsize=(4 * len(projections), 4))
for t, (ax, proj) in enumerate(zip(np.atleast_1d(axes), projections)):
    ax.imshow(proj, cmap='gray')
    ax.set_title(f't = {t}')
    ax.axis('off')
plt.show()

## Persist the projection to HDF5

Save the trained map + projection to a `_uv_projection.h5` file (the schema the flow
methods consume), then reload it to confirm the round-trip.

In [ ]:
from seamless import ProjectedFrame, save_projection_h5, FlowEstimator
from pathlib import Path

Path('outputs').mkdir(exist_ok=True)
frame0 = ProjectedFrame.from_parameterizer(param, volumes[t0], t=t0, uv_res=512)
save_projection_h5('outputs/ellipsoid_projection.h5', [frame0], topology=topology)
print('saved outputs/ellipsoid_projection.h5')

# Reload (import check): a fresh FlowEstimator builds straight from the file.
est = FlowEstimator.from_projection_h5('outputs/ellipsoid_projection.h5')
print('reloaded frames:', len(est.frames),
      '| NuvoMLP restored:', est.frames[0].nuvo_model is not None)

## Summary

With the high-level API the whole per-frame workflow is just:

```python
param = Parameterizer.from_volume(volume, topology='cylinder')
param.train()                                 # or train(base_model=prev) to warm-start
uv_map = param.get_uv_map()
multilayer = param.project_to_uv_along_normals(volume)
max_projection = multilayer.max(axis=0)
```

The `Parameterizer` handles surface extraction, normalization, normal estimation, NuvoMLP training, and volumetric projection — replacing ~100 lines of the original `nuvo_projection_timeseries.py`.